# 05 — Results summary

Parses the logs produced by 02/03/04 and compares them against the paper's Table 1 / Table 3 numbers. Writes a CSV to Drive and prints a markdown table you can paste into the BTP report.

In [ ]:
import os, re, glob, csv
BASE = '/content/drive/MyDrive/RiemannGFM/baseline'

PAPER = {
    'nc': {
        'citeseer':      {'acc': 66.38, 'f1': 66.41},
        'pubmed':        {'acc': 76.20, 'f1': 75.83},
        'github':        {'acc': 85.96, 'f1': 85.57},
        'airports_usa':  {'acc': 55.29, 'f1': 53.27},
    },
    'lp': {
        'citeseer':      {'auc': 99.40, 'ap': 98.42},
        'pubmed':        {'auc': 94.12, 'ap': 91.64},
        'github':        {'auc': 89.18, 'ap': 93.52},
        'airports_usa':  {'auc': 93.68, 'ap': 96.07},
    },
}

def _last_float(log_text, key):
    """Return the last float following `key=` or `key:` in the log."""
    pat = re.compile(rf'\b{re.escape(key)}\s*[:=]\s*([0-9]+\.[0-9]+)', re.IGNORECASE)
    m = pat.findall(log_text)
    return float(m[-1]) if m else None

rows = []

# ---- NC ----
for ds, tgt in PAPER['nc'].items():
    log = f'{BASE}/results/nc/{ds}.log'
    if not os.path.exists(log):
        continue
    txt = open(log).read()
    acc = _last_float(txt, 'acc') or _last_float(txt, 'accuracy')
    f1  = _last_float(txt, 'f1')  or _last_float(txt, 'weighted_f1')
    rows.append({'task':'NC','dataset':ds,'metric':'ACC','ours':acc,'paper':tgt['acc']})
    rows.append({'task':'NC','dataset':ds,'metric':'F1', 'ours':f1, 'paper':tgt['f1']})

# ---- LP ----
for ds, tgt in PAPER['lp'].items():
    log = f'{BASE}/results/lp/{ds}.log'
    if not os.path.exists(log):
        continue
    txt = open(log).read()
    auc = _last_float(txt, 'auc')
    ap  = _last_float(txt, 'ap')  or _last_float(txt, 'average_precision')
    rows.append({'task':'LP','dataset':ds,'metric':'AUC','ours':auc,'paper':tgt['auc']})
    rows.append({'task':'LP','dataset':ds,'metric':'AP', 'ours':ap, 'paper':tgt['ap']})

# Add delta and write CSV.
for r in rows:
    if r['ours'] is not None:
        r['delta'] = round(r['ours'] - r['paper'], 2)

out_csv = f'{BASE}/results/table1_comparison.csv'
with open(out_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['task','dataset','metric','ours','paper','delta'])
    w.writeheader()
    for r in rows: w.writerow(r)
print(f'Wrote {out_csv}')

# Pretty markdown.
print('\n| Task | Dataset | Metric | Ours | Paper | Δ |')
print('|------|---------|--------|-----:|------:|--:|')
for r in rows:
    ours = f"{r['ours']:.2f}" if r.get('ours') is not None else 'n/a'
    delta = f"{r.get('delta', 0):+.2f}" if 'delta' in r else 'n/a'
    print(f"| {r['task']} | {r['dataset']} | {r['metric']} | {ours} | {r['paper']:.2f} | {delta} |")